# The Competition Discount — Step 1: Get the data
**Source:** US DOT Domestic Airfare Consumer Report, Table 6 (all US city-pair markets with 10+ passengers/day), via data.transportation.gov (dataset `yj5y-b2ir`).
**Window:** 2019 Q1 → 2026 Q1.

Run the cells one at a time (not *Run all*). Everything is saved to Google Drive, so a runtime reset doesn't cost a re-download.


In [ ]:
# 1. Setup: save everything to Google Drive so a runtime reset doesn't lose the download
from google.colab import drive
drive.mount('/content/drive')

import os, io, time
import requests
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/the-competition-discount'
RAW_DIR = f'{PROJECT_DIR}/data/raw'
os.makedirs(RAW_DIR, exist_ok=True)

BASE = 'https://data.transportation.gov/resource/yj5y-b2ir'   # DOT Consumer Airfare Report, Table 6
START_YEAR = 2019
RAW_PATH = f'{RAW_DIR}/table6_{START_YEAR}_onward_raw.csv'
print('Saving to:', RAW_DIR)

In [ ]:
# 2. Peek: look at 5 rows and the column names before downloading anything big
r = requests.get(f'{BASE}.json', params={'$limit': 5}, timeout=60)
r.raise_for_status()
peek = pd.DataFrame(r.json())
print('Columns:', list(peek.columns))
peek

In [ ]:
# 3. How many rows per year? A cheap count on the server before the real download
r = requests.get(f'{BASE}.json',
                 params={'$select': 'year, count(*) AS rows', '$group': 'year', '$order': 'year'},
                 timeout=60)
r.raise_for_status()
by_year = pd.DataFrame(r.json())
print(by_year.tail(12).to_string(index=False))

In [ ]:
# 4. Download 2019 onward in pages of 50,000 rows, then save the raw file (kept as text on purpose)
PAGE = 50_000

def get_page(where, offset):
    params = {'$where': where, '$limit': PAGE, '$offset': offset, '$order': ':id'}
    r = requests.get(f'{BASE}.csv', params=params, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), dtype=str)

# 'year' may be stored as a number or as text on the portal, so try both filters
where = f'year >= {START_YEAR}'
try:
    first = get_page(where, 0)
except requests.HTTPError:
    where = f"year >= '{START_YEAR}'"
    first = get_page(where, 0)
print('Filter used:', where)

pages, offset = [first], 0
print(f'offset {offset:>7,}: {len(first):,} rows')
while len(pages[-1]) == PAGE:
    offset += PAGE
    time.sleep(1)                      # be polite to the public API
    pages.append(get_page(where, offset))
    print(f'offset {offset:>7,}: {len(pages[-1]):,} rows')

raw = pd.concat(pages, ignore_index=True)
raw.to_csv(RAW_PATH, index=False)
print(f'\nSaved {len(raw):,} rows to {RAW_PATH}')

In [ ]:
# 5. Checkpoint reload: after a runtime reset, start here instead of downloading again
raw = pd.read_csv(RAW_PATH, dtype=str)
print(f'{len(raw):,} rows, {raw.shape[1]} columns')

In [ ]:
# 6. Validation checkpoint: rows per quarter, missing values, duplicates
df = raw.copy()
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['quarter'] = pd.to_numeric(df['quarter'], errors='coerce')

per_q = df.groupby(['year', 'quarter']).size().rename('rows').reset_index()
print('Rows per quarter:')
print(per_q.to_string(index=False))
print('\nQuarters found:', len(per_q), '| first:', tuple(per_q.iloc[0, :2]), '| last:', tuple(per_q.iloc[-1, :2]))

print('\nMissing values per column:')
print(df.isna().sum().to_string())

# One row should exist per route per quarter; check on whichever route keys this table uses
key = [c for c in ['year', 'quarter', 'citymarketid_1', 'citymarketid_2'] if c in df.columns]
print('\nDuplicate route-quarter rows on', key, ':', df.duplicated(key).sum())